In [1]:
from copy import deepcopy

import fasttext
import fasttext.util
import matplotlib.pyplot as plt
from matplotlib.image import imread
from mpl_toolkits import mplot3d
from matplotlib import gridspec
from PIL import Image
import io
import os
from urllib.request import urlopen
from skimage.segmentation import mark_boundaries
from nltk.tokenize import RegexpTokenizer
from torchinfo import summary
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import requests
from scipy.stats import norm
import torch

from sklearn.metrics import classification_report
from torch.utils.tensorboard import SummaryWriter

from torchvision import datasets, transforms

2026-03-23 15:31:58.612427: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Код для обучения

In [4]:
class callback():
    def __init__(self, writer, dataset, loss_function, id2tag, ind_to_word, delimeter = 100, batch_size=64, num_exampels=3):
        self.step = 0
        self.writer = writer
        self.delimeter = delimeter
        self.loss_function = loss_function
        self.batch_size = batch_size
        self.num_exampels = num_exampels

        self.dataset = dataset
        self.id2tag = id2tag
        self.ind_to_word = ind_to_word

    def forward(self, model, loss):
        self.step += 1
        self.writer.add_scalar('LOSS/train', loss, self.step)
        
        if self.step % self.delimeter == 0:
            
            batch_generator = torch.utils.data.DataLoader(dataset = self.dataset, 
                                                          batch_size=self.batch_size, shuffle=True)
            
            pred = []
            real = []
            test_loss = 0
            model.eval()

            for it, (x_batch, y_batch) in enumerate(batch_generator):
                x_batch = x_batch.to(model.device)
                y_batch = y_batch.to(model.device)

                output = model(x_batch)

                test_loss += self.loss_function(output, y_batch).cpu().item() * len(x_batch)
            
            test_loss /= len(self.dataset)
            
            self.writer.add_scalar('LOSS/test', test_loss, self.step)

            x_batch, y_batch = next(iter(batch_generator))
            x_batch = x_batch.to(model.device)
            with torch.no_grad():
                outputs = model(x_batch)
                preds = torch.argmax(outputs, dim=1)

            text = f"Examples at step {self.step}\n"
            for i in range(self.num_exampels):
                tokens = [self.ind_to_word.get(idx.item(), '[UNK]') for idx in x_batch[i]]
                true_tags = [self.id2tag.get(idx.item(), '?') for idx in y_batch[i] if idx != -100]

                pred_tags = []
                for j, idx in enumerate(y_batch[i]):
                    if idx != -100:
                        pred_tags.append(self.id2tag.get(preds[i][j].item(), '?'))
                text += f"\nExample {i+1}:\n"
                text += f"Tokens: {' '.join(tokens)}\n"
                text += f"True:   {' '.join(true_tags)}\n"
                text += f"Pred:   {' '.join(pred_tags)}\n"

            self.writer.add_text('Examples', text, self.step)


          
    def __call__(self, model, loss):
        return self.forward(model, loss)

In [5]:
def train_on_batch(model, x_batch, y_batch, optimizer, loss_function):
    model.train()
    optimizer.zero_grad()
    
    output = model(x_batch.to(model.device))
    
    loss = loss_function(output, y_batch.to(model.device))
    loss.backward()

    optimizer.step()
    return loss.cpu().item()

In [6]:
def train_epoch(train_generator, model, loss_function, optimizer, callback = None):
    epoch_loss = 0
    total = 0
    for it, (batch_of_x, batch_of_y) in enumerate(train_generator):
        batch_loss = train_on_batch(model, batch_of_x, batch_of_y, optimizer, loss_function)
        
        if callback is not None:
            with torch.no_grad():
                callback(model, batch_loss)
            
        epoch_loss += batch_loss*len(batch_of_x)
        total += len(batch_of_x)
    
    return epoch_loss/total

In [7]:
def trainer(count_of_epoch, 
            batch_size, 
            dataset,
            model, 
            loss_function,
            optimizer,
            lr = 0.001,
            callback = None):

    optima = optimizer(model.parameters(), lr=lr)
    
    iterations = tqdm(range(count_of_epoch), desc='epoch')
    iterations.set_postfix({'train epoch loss': np.nan})
    for it in iterations:
        batch_generator = tqdm(
            torch.utils.data.DataLoader(dataset=dataset, 
                                        batch_size=batch_size, 
                                        shuffle=True, pin_memory=True), 
            leave=False, total=len(dataset)//batch_size+(len(dataset)%batch_size>0))
        
        epoch_loss = train_epoch(train_generator=batch_generator, 
                    model=model, 
                    loss_function=loss_function, 
                    optimizer=optima, 
                    callback=callback)
        
        iterations.set_postfix({'train epoch loss': epoch_loss})

In [8]:
def testing_on_test_sample(model, dataset_test_pt):
    batch_generator = torch.utils.data.DataLoader(dataset=dataset_test_pt, 
                                                  batch_size=64, 
                                                  pin_memory=True)
    pred = []
    real = []
    model.eval()
    for x_batch, y_batch in batch_generator:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        with torch.no_grad():
            output = model(x_batch)
        preds = torch.argmax(output, dim=1)
        mask = (y_batch != -100)
        pred.extend(preds[mask].cpu().numpy())
        real.extend(y_batch[mask].cpu().numpy())
    print(classification_report(real, pred, zero_division=0))

## Загрузка датасета

In [9]:
import gzip
from conllu import parse_incr

dataset = []
dataset_len = 100000
upos_to_ind = {}

with open('nerus_lenta.conllu', 'rt', encoding='utf-8') as f:
    count = 0
    for sent in tqdm(parse_incr(f), total=dataset_len, desc='Загрузка предложений'):
        text = sent.metadata['text']
        dataset.append([text, []]) # текст и ответ - размеченые части речи
        for token in sent:
            if token['upos'] not in upos_to_ind:
                upos_to_ind[token['upos']] = upos_to_ind.__len__()
            dataset[len(dataset) - 1][1].append(upos_to_ind[token['upos']])

        if count >= dataset_len:
            break
        count += 1

Загрузка предложений:   0%|          | 0/100000 [00:00<?, ?it/s]

In [10]:
import psutil
mem = psutil.virtual_memory()

print(f"Всего: {mem.total / (1024**3):.2f} GB")
print(f"Доступно: {mem.available / (1024**3):.2f} GB")
print(f"Используется: {mem.used / (1024**3):.2f} GB")
print(f"Процент использования: {mem.percent}%")

Всего: 15.43 GB
Доступно: 10.01 GB
Используется: 5.42 GB
Процент использования: 35.1%


In [11]:
print(dataset[:1])
print(len(upos_to_ind))
print(upos_to_ind)

[['Вице-премьер по социальным вопросам Татьяна Голикова рассказала, в каких регионах России зафиксирована наиболее высокая смертность от рака, сообщает РИА Новости.', [0, 1, 2, 0, 3, 3, 4, 5, 1, 6, 0, 3, 4, 7, 2, 0, 1, 0, 5, 4, 3, 3, 5]]]
17
{'NOUN': 0, 'ADP': 1, 'ADJ': 2, 'PROPN': 3, 'VERB': 4, 'PUNCT': 5, 'DET': 6, 'ADV': 7, 'PRON': 8, 'CCONJ': 9, 'SCONJ': 10, 'NUM': 11, 'PART': 12, 'AUX': 13, 'X': 14, 'SYM': 15, 'INTJ': 16}


#### Итого у нас 17 частей речи есть датасет, вида [предложение, [части речи слов]]

In [12]:
from sklearn.model_selection import train_test_split

dataset_train, dataset_test = train_test_split(dataset, test_size=0.2, random_state=42)

In [13]:
max_length_sentence = 20
num_classes = len(upos_to_ind)
print(num_classes, max_length_sentence)

17 20


## Создаём Rnn модель

In [14]:
import GPUtil; [print(f'GPU {g.id}: {g.memoryUsed:.0f} MB / {g.memoryTotal:.0f} MB') for g in GPUtil.getGPUs()]

GPU 0: 9 MB / 6144 MB


[None]

In [15]:
from gensim.models import KeyedVectors

model1 = KeyedVectors.load_word2vec_format('cc.ru.300.vec', binary=False)

word_to_ind = {}
matrix_fasttext = []

for i, w in enumerate(tqdm(model1.index_to_key, desc="Загрузка векторов")):
    v = model1.vectors[i]
    word_to_ind[w] = i
    matrix_fasttext.append(v)

special_tokens = ['[PAD]', '[UNK]', '[CLS]', '[SEP]']
for w in special_tokens:
    word_to_ind[w] = len(matrix_fasttext)
    matrix_fasttext.append(np.zeros_like(matrix_fasttext[-1]))

matrix_fasttext = torch.tensor(matrix_fasttext)

Загрузка векторов:   0%|          | 0/2000000 [00:00<?, ?it/s]

In [16]:
mem = psutil.virtual_memory()

print(f"Всего: {mem.total / (1024**3):.2f} GB")
print(f"Доступно: {mem.available / (1024**3):.2f} GB")
print(f"Используется: {mem.used / (1024**3):.2f} GB")
print(f"Процент использования: {mem.percent}%")

Всего: 15.43 GB
Доступно: 3.37 GB
Используется: 12.06 GB
Процент использования: 78.2%


In [17]:
class Tokenizer(object):
    def __init__(self, word_to_ind, tokenizer):
        self.word_to_ind = word_to_ind
        self.tokenizer = tokenizer
    def __call__(self, sentences, max_length=None, pad_to_max_length=True):
        if max_length is None:
            max_length = max_length_sentence - 2
        tokens = self.tokenizer.tokenize_sents(sentences)
        processed = []
        for s in tokens:
            s = s[:max_length]
            sent_tokens = ['[CLS]'] + s + ['[SEP]']
            pad_len = max_length_sentence - len(sent_tokens)
            if pad_len > 0:
                sent_tokens += ['[PAD]'] * pad_len
            processed.append(sent_tokens)
        ids = [[self.word_to_ind.get(w, self.word_to_ind['[UNK]']) for w in sent] for sent in processed]
        return torch.tensor(ids)

In [18]:
tokenizer = Tokenizer(word_to_ind, RegexpTokenizer(r'[а-яА-ЯёЁ]+|[^\w\s]|\d+'))

train_texts = [item[0] for item in dataset_train]
test_texts  = [item[0] for item in dataset_test]

train_data_sent = tokenizer(train_texts)
test_data_sent  = tokenizer(test_texts)


def prepare_labels(original_labels, max_len):
    labels = original_labels[:max_len - 2]
    labels_with_special = [-100] + labels + [-100]
    pad_len = max_len - len(labels_with_special)
    if pad_len > 0:
        labels_with_special += [-100] * pad_len
    return labels_with_special

train_y = [prepare_labels(item[1], max_length_sentence) for item in dataset_train]
test_y  = [prepare_labels(item[1], max_length_sentence) for item in dataset_test]

dataset_train_pt = torch.utils.data.TensorDataset(train_data_sent, torch.tensor(train_y))
dataset_test_pt  = torch.utils.data.TensorDataset(test_data_sent,  torch.tensor(test_y))

In [19]:
print(f"Num classes: {num_classes}")
print(f"Num classes: {np.unique(train_y)}")
print(test_y[:10])
print(train_data_sent[:10])

Num classes: 17
Num classes: [-100    0    1    2    3    4    5    6    7    8    9   10   11   12
   13   14   15   16]
[[-100, 5, 2, 0, 5, 1, 3, 4, 0, 5, 7, 10, 8, 4, 0, 5, 0, 0, 9, -100], [-100, 3, 4, 0, 1, 8, 5, 10, 0, 5, 4, 1, 0, 5, 5, 9, 0, 4, 13, -100], [-100, 0, 0, 4, 1, 6, 0, 2, 0, 3, 3, 5, -100, -100, -100, -100, -100, -100, -100, -100], [-100, 0, 4, 2, 0, 5, 2, 0, 5, 5, 10, 4, 0, 0, 5, 10, 1, 0, 2, -100], [-100, 0, 4, 4, 0, 5, 9, 12, 4, 8, 4, 5, -100, -100, -100, -100, -100, -100, -100, -100], [-100, 3, 3, 6, 0, 1, 0, 5, 3, 5, 12, 4, 5, -100, -100, -100, -100, -100, -100, -100], [-100, 1, 0, 3, 4, 2, 9, 2, 5, 8, 6, 0, 4, 7, 5, -100, -100, -100, -100, -100], [-100, 2, 0, 3, 4, 1, 2, 0, 5, 0, 5, 5, 3, 5, 5, 4, 0, 1, 0, -100], [-100, 1, 0, 7, 13, 7, 4, 0, 0, 9, 0, 4, 7, 0, 5, -100, -100, -100, -100, -100], [-100, 7, 8, 4, 0, 9, 4, 12, 4, 1, 0, 2, 11, 0, 5, -100, -100, -100, -100, -100]]
tensor([[2000002,  248910,   60139,      10,    5155,    3055,       0,     156,
          

In [20]:
# Получаем статистику памяти
mem = psutil.virtual_memory()

print(f"Всего: {mem.total / (1024**3):.2f} GB")
print(f"Доступно: {mem.available / (1024**3):.2f} GB")
print(f"Используется: {mem.used / (1024**3):.2f} GB")
print(f"Процент использования: {mem.percent}%")

Всего: 15.43 GB
Доступно: 3.35 GB
Используется: 12.08 GB
Процент использования: 78.3%


In [21]:
import gc
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [22]:
class RNN_2(torch.nn.Module):
    @property
    def device(self):
        return next(self.parameters()).device

    def __init__(self, vocab_dim, max_len, num_tags = 17, emb_dim=10, hidden_dim=10,
                 num_layers=3, bidirectional=False, p=0.7, use_batchnorm=False):
        super(RNN_2, self).__init__()
        self.num_tags = num_tags
        self.max_len = max_len
        self.vocab_dim = vocab_dim
        pad_idx = word_to_ind['[PAD]']
        self.embedding = torch.nn.Embedding(vocab_dim, emb_dim, padding_idx=pad_idx)
        self.encoder = torch.nn.LSTM(emb_dim, hidden_dim, num_layers,
                                     bidirectional=bidirectional,
                                     batch_first=True, dropout=p)
        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)
        self.use_batchnorm = use_batchnorm
        if use_batchnorm:
            self.batchnorm = torch.nn.BatchNorm1d(lstm_out_dim)

        self.linear1 = torch.nn.Linear(lstm_out_dim, lstm_out_dim)
        self.linear2 = torch.nn.Linear(lstm_out_dim, num_tags)

    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.encoder(emb)
        if self.use_batchnorm:
            lstm_out = lstm_out.transpose(1, 2)
            lstm_out = self.batchnorm(lstm_out)
            lstm_out = lstm_out.transpose(1, 2)

        logits = self.linear2(torch.relu(self.linear1(lstm_out)))

        return logits.permute(0, 2, 1)

In [23]:
config = {
    'vocab_dim': len(word_to_ind),
    'num_tags': num_classes,
    'max_len': max_length_sentence,
    'emb_dim': 300,
    'hidden_dim': 64,
    'num_layers': 3,
    'bidirectional': True,
    'p': 0.7,
    'use_batchnorm': True,
}


model = RNN_2(**config)
_ = model.to(device)

model.embedding.weight.data.copy_(matrix_fasttext)
for param in model.embedding.parameters():
    param.requires_grad = False
model.to(device)

summary(model, input_size=(1, max_length_sentence), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
RNN_2                                    [1, 17, 20]               --
├─Embedding: 1-1                         [1, 20, 300]              (600,001,200)
├─LSTM: 1-2                              [1, 20, 128]              386,048
├─BatchNorm1d: 1-3                       [1, 128, 20]              256
├─Linear: 1-4                            [1, 20, 128]              16,512
├─Linear: 1-5                            [1, 20, 17]               2,193
Total params: 600,406,209
Trainable params: 405,009
Non-trainable params: 600,001,200
Total mult-adds (Units.MEGABYTES): 607.74
Input size (MB): 0.00
Forward/backward pass size (MB): 0.11
Params size (MB): 2401.62
Estimated Total Size (MB): 2401.74

In [24]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     80516
           1       0.00      0.00      0.00     34671
           2       0.00      0.00      0.00     26326
           3       0.00      0.00      0.00     21025
           4       0.00      0.00      0.00     35480
           5       0.00      0.00      0.00     45034
           6       0.00      0.00      0.00      4813
           7       0.00      0.00      0.00      8062
           8       0.00      0.00      0.00     11015
           9       0.00      0.00      0.00      6231
          10       0.02      1.00      0.04      5396
          11       0.00      0.00      0.00      5556
          12       0.00      0.00      0.00      3531
          13       0.00      0.00      0.00      2094
          14       0.00      0.00      0.00      2697
          15       0.00      0.00      0.00        67
          16       0.00      0.00      0.00         8

    accuracy              

In [25]:
import gc
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   2299 MiB |   2365 MiB |  24008 MiB |  21709 MiB |
|       from large pool |   2298 MiB |   2363 MiB |  22664 MiB |  20366 MiB |
|       from small pool |      1 MiB |      3 MiB |   1343 MiB |   1342 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   2299 MiB |   2365 MiB |  24008 MiB |  21709 MiB |
|       from large pool |   2298 MiB |   2363 MiB |  22664 MiB |

In [26]:
import GPUtil; [print(f'GPU {g.id}: {g.memoryUsed:.0f} MB / {g.memoryTotal:.0f} MB') for g in GPUtil.getGPUs()]

GPU 0: 2415 MB / 6144 MB


[None]

In [27]:
train_labels = dataset_train_pt.tensors[1]
valid_labels = train_labels[train_labels != -100]

real_classes = torch.unique(valid_labels)
num_classes = len(real_classes)

counts = torch.bincount(valid_labels)

weights = 1.0 / torch.sqrt(counts.float())
weights = weights / weights.mean()
weights = weights.to(device)

loss_function = torch.nn.CrossEntropyLoss(
    ignore_index=-100,
    weight=weights
)
optimizer = torch.optim.Adam

In [28]:
writer = SummaryWriter(log_dir = 'model-lstm-1')

ind_to_word = {v: k for k, v in word_to_ind.items()}
id2tag = {v: k for k, v in upos_to_ind.items()}
# (self, writer, dataset, loss_function, id2tag, ind_to_word, delimeter = 100, batch_size=64, num_exampels=3):
call = callback(writer, dataset_test_pt, loss_function, id2tag, ind_to_word, delimeter = 200)

trainer(count_of_epoch=1, 
        batch_size=64, 
        dataset=dataset_train_pt,
        model=model, 
        loss_function=loss_function,
        optimizer = optimizer,
        lr=0.0005,
        callback=call)

epoch:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1250 [00:00<?, ?it/s]

In [29]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.86      0.90      0.88     80516
           1       0.93      0.87      0.90     34671
           2       0.84      0.76      0.80     26326
           3       0.74      0.78      0.76     21025
           4       0.87      0.84      0.85     35480
           5       0.88      0.87      0.87     45034
           6       0.79      0.81      0.80      4813
           7       0.79      0.80      0.79      8062
           8       0.87      0.87      0.87     11015
           9       0.82      0.85      0.83      6231
          10       0.84      0.89      0.86      5396
          11       0.68      0.79      0.73      5556
          12       0.84      0.78      0.81      3531
          13       0.83      0.89      0.86      2094
          14       0.36      0.33      0.35      2697
          15       0.61      0.60      0.60        67
          16       0.78      0.88      0.82         8

    accuracy              

In [30]:
!kill $(pgrep tensorboard)
!tensorboard --logdir model-lstm-1

kill: использование: kill [-s назв_сигнала | -n номер_сигнала | -назв_сигнала] ид_процесса | назв_задания] ... или kill -l [назв_сигнала]
/home/sasha/Documents/venv/lib/python3.12/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-03-23 15:42:34.111985: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Serving 